In [1]:
import os

os.environ[
    "TF_USE_LEGACY_KERAS"
] = "1"

In [2]:
!pip install -q tensorflow-recommenders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 2.7 MB/s eta 0:00:00a 0:00:01


In [3]:
import pandas as pd
import pickle

import tensorflow as tf
import tensorflow_recommenders as tfrs

import numpy as np

2026-05-26 04:21:14.265601: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779769274.485340      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779769274.554823      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779769275.073285      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779769275.073333      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779769275.073337      57 computation_placer.cc:177] computation placer alr

In [4]:
training_df = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/train-ds/training_dataset.parquet"
)

with open(
    "/kaggle/input/datasets/selinparmar/feature-vocab/feature_vocab.pkl",
    "rb"
) as file:

    feature_vocab = pickle.load(file)

In [5]:
training_df = (
    training_df
    .sample(
        200000,
        random_state=42
    )
)

print(
    training_df.shape
)

(200000, 16)


In [6]:
training_df[
    'customer_id'
] = (
    training_df[
        'customer_id'
    ].astype(str)
)

training_df[
    'article_id'
] = (
    training_df[
        'article_id'
    ].astype(str)
)

In [7]:
user_vocab = (
    feature_vocab[
        'user_vocab'
    ]
)

favorite_product_vocab = (
    feature_vocab[
        'favorite_product_vocab'
    ]
)

item_vocab = (
    feature_vocab[
        'item_vocab'
    ]
)

product_type_vocab = (
    feature_vocab[
        'product_type_vocab'
    ]
)

season_vocab = (
    feature_vocab[
        'season_vocab'
    ]
)

In [8]:
class QueryTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Customer ID embedding
        self.customer_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=user_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(user_vocab) + 1,
                32
            )
        ])

        # Favorite product type embedding
        self.favorite_product_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                favorite_product_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    favorite_product_vocab
                ) + 1,
                16
            )
        ])

        # Season embedding
        self.season_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                season_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    season_vocab
                ) + 1,
                8
            )
        ])

        # Dense network
        self.dense_layers = tf.keras.Sequential([

            tf.keras.layers.Dense(
                128,
                activation='relu'
            ),

            tf.keras.layers.Dense(64)
        ])

    def call(self, features):

        customer_vector = (
            self.customer_embedding(
                features['customer_id']
            )
        )

        favorite_product_vector = (
            self.favorite_product_embedding(
                features[
                    'favorite_product_type'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features['season']
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features['age'],
                tf.float32
            ),

            tf.cast(
                features[
                    'purchase_frequency'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'avg_spend'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'repeat_purchase_ratio'
                ],
                tf.float32
            )

        ], axis=1)

        return self.dense_layers(
            tf.concat([

                customer_vector,
                favorite_product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

In [9]:
class CandidateTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Article embedding
        self.article_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=item_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(item_vocab) + 1,
                    32
                )
            ])
        )

        # Product type embedding
        self.product_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    product_type_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        product_type_vocab
                    ) + 1,
                    16
                )
            ])
        )

        # Season embedding
        self.season_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    season_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        season_vocab
                    ) + 1,
                    8
                )
            ])
        )

        # Dense network
        self.dense_layers = (
            tf.keras.Sequential([

                tf.keras.layers.Dense(
                    128,
                    activation='relu'
                ),

                tf.keras.layers.Dense(
                    64
                )
            ])
        )

    def call(
        self,
        features
    ):

        article_vector = (
            self.article_embedding(
                features[
                    'article_id'
                ]
            )
        )

        product_vector = (
            self.product_embedding(
                features[
                    'product_type_name'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features[
                    'season'
                ]
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features[
                    'purchase_count'
                ],
                tf.float32
            )

        ], axis=1)

        combined_features = (
            tf.concat([

                article_vector,
                product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

        return (
            self.dense_layers(
                combined_features
            )
        )

In [10]:
#candidate dataset
candidate_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        {
            "article_id":
                training_df[
                    "article_id"
                ],

            "product_type_name":
                training_df[
                    "product_type_name"
                ],

            "season":
                training_df[
                    "season"
                ],

            "purchase_count":
                training_df[
                    "purchase_count"
                ]
                .astype(float)
        }
    )
    .batch(256)
)

2026-05-26 04:22:08.389654: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [11]:
class TwoTowerModel(
    tfrs.models.Model
):

    def __init__(self):

        super().__init__()

        self.query_model = (
            QueryTower()
        )

        self.candidate_model = (
            CandidateTower()
        )

        self.task = (
            tfrs.tasks.Retrieval(
                metrics=
                tfrs.metrics.FactorizedTopK(

                    candidates=
                    candidate_dataset.map(
                        self.candidate_model
                    )
                )
            )
        )

    def compute_loss(
        self,
        features,
        training=False
    ):

        query_embeddings = (
            self.query_model(
                features
            )
        )

        candidate_embeddings = (
            self.candidate_model(
                features
            )
        )

        return self.task(
            query_embeddings,
            candidate_embeddings
        )

In [12]:
model = (
    TwoTowerModel()
)

model.compile(
    optimizer=
    tf.keras.optimizers.Adagrad(
        learning_rate=0.1
    )
)

In [13]:
for batch in tf.data.Dataset.from_tensor_slices(
    {
        key:
        training_df[
            key
        ].head(1).values

        for key in
        training_df.columns
    }
).batch(1):

    model.compute_loss(
        batch,
        training=False
    )

print(
    "Model fully built!"
)

/usr/local/lib/python3.12/dist-packages/tensorflow/python/util/dispatch.py:1260: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(1, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return dispatch_target(*args, **kwargs)


Model fully built!


In [14]:
model.load_weights(
    "/kaggle/input/datasets/selinparmar/two-tower-model-sample/two_tower_weights.weights.h5"
)

print(
    "Final weights loaded!"
)

Final weights loaded!


In [15]:
model.query_model.save_weights(
    "/kaggle/working/query_tower.weights.h5"
)

model.candidate_model.save_weights(
    "/kaggle/working/candidate_tower.weights.h5"
)

print(
    "Tower weights exported!"
)

Tower weights exported!


In [16]:
unique_items = (

    training_df[

        [
            'article_id',
            'product_type_name',
            'season',
            'purchase_count'
        ]

    ]

    .drop_duplicates()
)

In [18]:
item_inputs = {

    'article_id':
        tf.constant(
            unique_items[
                'article_id'
            ].astype(str).values
        ),

    'product_type_name':
        tf.constant(
            unique_items[
                'product_type_name'
            ].astype(str).values
        ),

    'season':
        tf.constant(
            unique_items[
                'season'
            ].astype(str).values
        ),

    'purchase_count':
        tf.constant(
            unique_items[
                'purchase_count'
            ].values,
            dtype=tf.float32
        )
}

all_embeddings = (
    model.candidate_model(
        item_inputs
    )
    .numpy()
)

item_embeddings_df = pd.DataFrame({

    "article_id":
        unique_items[
            'article_id'
        ].values,

    "embedding":
        list(
            all_embeddings
        )
})

print(
    "Item embeddings created!"
)

Item embeddings created!


In [19]:
item_embeddings_df.to_pickle(
    "/kaggle/working/item_embeddings.pkl"
)

print(
    "Item embeddings saved!"
)

Item embeddings saved!


In [20]:
item_embeddings_df.head()

,article_id,embedding
0,636455003,"[570.81226, 742.72394, 1537.7488, -128.62434, ..."
1,642895001,"[102.266685, 153.76515, 304.62927, -21.509987,..."
2,710676001,"[12.76776, 43.99569, 57.53849, 2.7059765, 6.78..."
3,827968002,"[243.49612, 338.93417, 700.4439, -54.699238, -..."
4,744862001,"[245.73427, 331.8509, 686.46985, -52.14904, -1..."
